# Clase 051 — Desafíos del ML: overfitting, underfitting, datos insuficientes

Diagnosticamos sub/overfitting mirando la brecha train-validation, visualizamos el
bias-variance tradeoff y usamos regularización (`Ridge`) como contramedida.

Requiere: `numpy`, `pandas`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, learning_curve, validation_curve
from sklearn.metrics import r2_score

np.random.seed(42)
print('setup ok')

## 1. Dataset no lineal con ruido

Generamos `y = sin(x) + ruido`. Un modelo lineal simple *underfittea*; un polinomio de grado
alto *overfittea*.

In [ ]:
rng = np.random.default_rng(42)
n = 80
X = np.sort(rng.uniform(-3, 3, n)).reshape(-1, 1)
y = np.sin(X).ravel() + rng.normal(0, 0.3, n)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
print('train', Xtr.shape, 'test', Xte.shape)

## 2. Overfitting: polinomio de grado 15

Train casi perfecto pero test malo (o negativo): el modelo memorizó el ruido.

In [ ]:
over = make_pipeline(PolynomialFeatures(15), StandardScaler(), LinearRegression())
over.fit(Xtr, ytr)
r2_tr_over = r2_score(ytr, over.predict(Xtr))
r2_te_over = r2_score(yte, over.predict(Xte))
print(f'grado 15  -> train R2 {r2_tr_over:.3f} | test R2 {r2_te_over:.3f}')
assert r2_tr_over - r2_te_over > 0.1, 'esperamos brecha train>>test (overfitting)'
print('brecha grande train-test = overfitting (baja bias, alta variance)')

## 3. Underfitting: grado 1 sobre datos no lineales

Ambos scores bajos y parecidos: el modelo es demasiado rígido para captar la curva.

In [ ]:
under = make_pipeline(PolynomialFeatures(1), LinearRegression())
under.fit(Xtr, ytr)
r2_tr_un = r2_score(ytr, under.predict(Xtr))
r2_te_un = r2_score(yte, under.predict(Xte))
print(f'grado 1   -> train R2 {r2_tr_un:.3f} | test R2 {r2_te_un:.3f}')
print('ambos bajos y parecidos = underfitting (alto bias)')

## 4. Regularización: Ridge sobre las mismas features polinomiales

`Ridge` penaliza la norma de los coeficientes: baja variance a cambio de algo de bias.
Debería mejorar el test respecto al grado 15 sin regularizar.

In [ ]:
reg = make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha=1.0))
reg.fit(Xtr, ytr)
r2_tr_reg = r2_score(ytr, reg.predict(Xtr))
r2_te_reg = r2_score(yte, reg.predict(Xte))
print(f'Ridge a=1 -> train R2 {r2_tr_reg:.3f} | test R2 {r2_te_reg:.3f}')
assert r2_te_reg > r2_te_over, 'Ridge deberia mejorar el test vs grado 15 sin regularizar'
print('la regularizacion recupera generalizacion')

## 5. Validation curve: score vs alpha (fuerza de regularización)

Barremos `alpha` para encontrar el sweet spot bias-variance.

In [ ]:
alphas = np.logspace(-3, 3, 12)
model = make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge())
tr_sc, va_sc = validation_curve(model, X, y, param_name='ridge__alpha',
                                param_range=alphas, cv=5, scoring='r2')
best_alpha = alphas[va_sc.mean(axis=1).argmax()]
print(f'mejor alpha por CV: {best_alpha:.4g}  (R2 val {va_sc.mean(axis=1).max():.3f})')

## 6. Learning curve: ¿la brecha se cierra con más datos?

Si `val_score` sigue subiendo al agregar datos, faltan datos. Si ambas convergen bajo,
el cuello de botella es el modelo/las features.

In [ ]:
sizes, tr_lc, va_lc = learning_curve(
    make_pipeline(PolynomialFeatures(4), StandardScaler(), Ridge(alpha=1.0)),
    X, y, train_sizes=np.linspace(0.2, 1.0, 6), cv=5, scoring='r2')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogx(alphas, tr_sc.mean(axis=1), 'o-', label='train')
axes[0].semilogx(alphas, va_sc.mean(axis=1), 's-', label='validation')
axes[0].axvline(best_alpha, color='k', ls='--', lw=0.8)
axes[0].set_xlabel('alpha (Ridge)'); axes[0].set_ylabel('R2')
axes[0].set_title('Validation curve'); axes[0].legend()

axes[1].plot(sizes, tr_lc.mean(axis=1), 'o-', label='train')
axes[1].plot(sizes, va_lc.mean(axis=1), 's-', label='validation')
axes[1].set_xlabel('tamano de train'); axes[1].set_ylabel('R2')
axes[1].set_title('Learning curve'); axes[1].legend()
plt.tight_layout(); plt.show()

## Ejercicios

1. Repetí el diagnóstico con `make_regression(n_samples=50, noise=20)` y un polinomio de
   grado 15. ¿Es overfitting o underfitting? Justificá con los R2 de train y test.
2. Barré `Ridge(alpha)` en `[0.001, 0.01, 0.1, 1, 10, 100]` sobre el dataset del ejercicio 1
   y graficá la curva de validación. ¿Dónde está el sweet spot?
3. Simulá sampling bias: entrená con un train desbalanceado 90/10 (clasificación binaria) y
   testeá en un test balanceado. ¿Qué te oculta el accuracy global?
4. Compará `Lasso` vs `Ridge` sobre las features polinomiales. ¿Cuántos coeficientes lleva
   Lasso a 0 exacto? ¿Qué implica para selección de features?

## Conclusiones

- Overfitting = train alto, test bajo (baja bias, alta variance). Underfitting = ambos bajos.
- La regularización (`Ridge`/`Lasso`) cambia variance por algo de bias y suele recuperar test.
- La validation curve encuentra el `alpha` óptimo; la learning curve dice si faltan datos.
- El test set se separa al principio y no se usa para tunear (eso va por CV en el train).